# Lab 3.2 – Diagramme, die eine Aussage haben

**Beispiel (fertige Lösung).** Zellen der Reihe nach mit **Umschalt+Enter** ausführen.

| Diagramm | Aussage | Tabelle aus Lab 3.1 | Entsteht |
| --- | --- | --- | --- |
| `fig1` | Verlauf | `kanal_monat` | fertig |
| `fig2` | Vergleich | `marge` | mit vorgegebenem Prompt |
| `fig3` | Abweichung | `wachstum` | mit eigenem Prompt |

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go   # nur für die Prüfzellen

QUELLE = "project/examples/m03-exploration/l01-eda/"   # Ergebnistabellen aus Lab 3.1 (Beispiel)
ORDNER = "project/examples/m03-exploration/l02-charts/"                    # hier landen die HTML-Diagramme

kanal_monat = pd.read_csv(QUELLE + "kanal_monat.csv")
marge = pd.read_csv(QUELLE + "marge_produktgruppe.csv")
wachstum = pd.read_csv(QUELLE + "wachstum_region.csv")
wachstum

,region,2024,2025,2026,wachstum_prozent
0,West,656796.9,721872.8,792757.2,9.8
1,Süd,594101.8,617795.6,674400.2,9.2
2,Ost,513457.4,477187.9,520396.9,9.1
3,Nord,750940.8,792446.5,826874.9,4.3


In [2]:
# Prüfhilfe – fertig, nicht ändern
def pruefe_diagramm(fig, name, tabelle, kategorie, wert):
    """OK, wenn fig ein plotly-Diagramm mit Titel ist und die Spalten kategorie und wert zeigt."""
    if not isinstance(fig, go.Figure):
        return f"nicht OK – {name} ist noch kein plotly-Diagramm."
    if not fig.layout.title.text:
        return f"nicht OK – {name} hat keinen Titel: den Befund als title=\"...\" setzen."
    soll_kategorien = set(tabelle[kategorie].astype(str))
    soll_werte = sorted(tabelle[wert].round(1))
    for spur in fig.data:
        for achse_a, achse_b in [(spur.x, spur.y), (spur.y, spur.x)]:
            if achse_a is None or achse_b is None or set(map(str, achse_a)) != soll_kategorien:
                continue
            if sorted(round(float(z), 1) for z in achse_b) == soll_werte:
                return f"OK – {name}: Spalten {kategorie} und {wert}, Titel „{fig.layout.title.text}“"
            return f"nicht OK – {name} zeigt {kategorie}, aber nicht die Werte der Spalte {wert}."
    return f"nicht OK – {name} zeigt nicht die Spalte {kategorie} ({', '.join(sorted(soll_kategorien))})."

## Diagramm 1 (fertig): Verlauf je Kanal

Liniendiagramm für einen Verlauf über die Zeit; der Titel nennt den Befund, nicht nur das Thema.

In [3]:
fig1 = px.line(
    kanal_monat, x="monat", y=["Außendienst", "Onlineshop"],
    title="Seit dem Relaunch im März 2026 wächst der Onlineshop, der Außendienst bleibt flach",
    labels={"monat": "Monat", "value": "Umsatz in €", "variable": "Kanal"},
)
fig1

## Diagramm 2: Vergleich der Produktgruppen

Balkendiagramm für einen Vergleich von Gruppen.

In [4]:
# >>> AUFGABE 1: Balkendiagramm fig2 aus der Tabelle marge – Prompt siehe Übung, Aufgabe 2
fig2 = px.bar(
    marge, x="produktgruppe", y="marge_prozent", text="marge_prozent",
    title="Schreibwaren mit der höchsten Marge (34,8 %), Technik mit der niedrigsten (14,2 %)",
    labels={"produktgruppe": "Produktgruppe", "marge_prozent": "Marge in %"},
)
# <<< ENDE AUFGABE 1
fig2

In [5]:
# Prüfzelle – nicht ändern
print(pruefe_diagramm(fig2, "fig2", marge, "produktgruppe", "marge_prozent"))

OK – fig2: Spalten produktgruppe und marge_prozent, Titel „Schreibwaren mit der höchsten Marge (34,8 %), Technik mit der niedrigsten (14,2 %)“


## Diagramm 3: Abweichung je Region

Balken für die Veränderung zum Vorjahr; der Titel nennt den Befund.

In [6]:
# >>> AUFGABE 2: eigenes Diagramm fig3 aus der Tabelle wachstum – eigener Prompt, siehe Übung, Aufgabe 4
fig3 = px.bar(
    wachstum, x="region", y="wachstum_prozent", text="wachstum_prozent",
    title="West, Süd und Ost wachsen um 9 bis 10 %, Nord nur um 4,3 % (Jan.–Sep. 2026 zum Vorjahr)",
    labels={"region": "Region", "wachstum_prozent": "Veränderung zum Vorjahr in %"},
)
# <<< ENDE AUFGABE 2
fig3

In [7]:
# Prüfzelle – nicht ändern
print(pruefe_diagramm(fig3, "fig3", wachstum, "region", "wachstum_prozent"))

OK – fig3: Spalten region und wachstum_prozent, Titel „West, Süd und Ost wachsen um 9 bis 10 %, Nord nur um 4,3 % (Jan.–Sep. 2026 zum Vorjahr)“


## Als HTML speichern

`write_html` schreibt ein interaktives Diagramm als Datei, die jeder Browser öffnet. `include_plotlyjs="cdn"` hält die Datei klein; plotly wird beim Öffnen aus dem Netz geladen.

In [8]:
diagramme = {"diagramm_kanal.html": fig1, "diagramm_marge.html": fig2, "diagramm_wachstum.html": fig3}
for datei, fig in diagramme.items():
    if isinstance(fig, go.Figure):
        fig.write_html(ORDNER + datei, include_plotlyjs="cdn")
        print("Gespeichert:", ORDNER + datei)
    else:
        print("Übersprungen (noch kein Diagramm):", datei)

Gespeichert: project/examples/m03-exploration/l02-charts/diagramm_kanal.html
Gespeichert: project/examples/m03-exploration/l02-charts/diagramm_marge.html
Gespeichert: project/examples/m03-exploration/l02-charts/diagramm_wachstum.html


## Erweiterung (Beispiel): Ist gegen Plan je Region

Abweichung = (Ist − Plan) / Plan, Januar bis September 2026. Ist aus `umsatz_features.csv` (Beispiel Lab 2.3), Plan aus `data/nordwind_plan_2026.csv`.

In [9]:
features = pd.read_csv("project/examples/m02-data-preparation/l03-features/umsatz_features.csv")
plan = pd.read_csv("data/nordwind_plan_2026.csv")

ist_2026 = features[features["monat"].between("2026-01", "2026-09")].groupby("region")["umsatz"].sum()
plan_2026 = plan[plan["monat"].between("2026-01", "2026-09")].groupby("region")["plan_umsatz"].sum()
abweichung = ((ist_2026 - plan_2026) / plan_2026 * 100).round(1).reset_index(name="abweichung_prozent")
abweichung["lage"] = abweichung["abweichung_prozent"].apply(lambda w: "über Plan" if w >= 0 else "unter Plan")

fig4 = px.bar(
    abweichung, x="region", y="abweichung_prozent", color="lage", text="abweichung_prozent",
    title="Ist gegen Plan, Januar bis September 2026: Abweichung je Region in %",
    labels={"region": "Region", "abweichung_prozent": "Abweichung zum Plan in %", "lage": ""},
)
fig4